# Task 1: Build a Reusable bootstrap_ci Function

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

In [2]:
def bootstrap_ci(data, stat_func=np.mean, n_boot=10_000, ci_level=0.95):
    """
    Returns (lower, upper) percentile bootstrap CI.

    Parameters
    ----------
    data : array-like
        The sample to resample from.
    stat_func : callable
        The statistic to compute on each resample (default: np.mean).
    n_boot : int
        Number of bootstrap resamples.
    ci_level : float
        Confidence level (e.g. 0.95 for a 95% CI).
    """

    data = np.asarray(data)
    boot_stats = np.zeros(n_boot)

    for i in range(n_boot):
        boot_sample = np.random.choice(data, size=len(data), replace=True)
        boot_stats[i] = stat_func(boot_sample)

    alpha = 1 - ci_level
    lower = np.percentile(boot_stats, 100 * alpha / 2)
    upper = np.percentile(boot_stats, 100 * (1 - alpha / 2))

    return lower, upper

test_data = np.arange(1, 101)
ci_lower, ci_upper = bootstrap_ci(test_data)
print(f"95% Bootstrap CI: [{ci_lower:.2f}, {ci_upper:.2f}]")

95% Bootstrap CI: [44.80, 56.10]


I tested the function on `np.arange(1, 101)` using the default mean statistic. Since the true mean of this data is 50.5, the bootstrap confidence interval should be centered around that value. The result came out roughly between 40 and 60, which shows that the function is working correctly and the bootstrap distribution is giving a reasonable estimate of uncertainty.

# Task 2: Apply Bootstrap CIs to Real Data


In [3]:
df = pd.DataFrame({
    "response_time": np.random.normal(loc=150, scale=30, size=300),
    "clicked": np.random.binomial(1, 0.12, size=300)
})

# Compute bootstrap CIs
mean_ci = bootstrap_ci(df["response_time"], stat_func=np.mean, n_boot=10_000, ci_level=0.95)
median_ci = bootstrap_ci(df["response_time"], stat_func=np.median, n_boot=10_000, ci_level=0.95)
prop_ci = bootstrap_ci(df["clicked"], stat_func=np.mean, n_boot=10_000, ci_level=0.95)

# Summary table
summary_table = pd.DataFrame({
    "Statistic": ["Mean", "Median", "Proportion"],
    "Column": ["response_time", "response_time", "clicked"],
    "CI Lower": [mean_ci[0], median_ci[0], prop_ci[0]],
    "CI Upper": [mean_ci[1], median_ci[1], prop_ci[1]]
})

summary_table

,Statistic,Column,CI Lower,CI Upper
0,Mean,response_time,149.020613,155.867277
1,Median,response_time,149.451899,156.234552
2,Proportion,clicked,0.063333,0.126667


I generated a synthetic dataset with 300 rows using the same style as the lesson examples. The dataset includes a continuous variable called `response_time`, generated from a normal distribution, and a binary variable called `clicked`, generated from a binomial distribution. Then I computed 95% bootstrap confidence intervals with 10,000 resamples for the mean and median of the continuous variable, and for the proportion of the binary variable. The results are summarized in the table below.

# Task 3: Compare Bootstrap CIs with Normal-Approximation CIs


In [4]:
mean_data = df["response_time"]
mean_sample = mean_data.mean()
mean_se = st.sem(mean_data)

mean_ci_normal = st.t.interval(
    confidence=0.95,
    df=len(mean_data) - 1,
    loc=mean_sample,
    scale=mean_se
)

# Proportion: classical 95% Wald interval
prop_data = df["clicked"]
p_hat = prop_data.mean()
n = len(prop_data)
z = 1.96
prop_se = np.sqrt(p_hat * (1 - p_hat) / n)

prop_ci_normal = (
    p_hat - z * prop_se,
    p_hat + z * prop_se
)

# Comparison table
comparison_table = pd.DataFrame({
    "Statistic": ["Mean", "Proportion"],
    "Bootstrap Lower": [mean_ci[0], prop_ci[0]],
    "Bootstrap Upper": [mean_ci[1], prop_ci[1]],
    "Normal Lower": [mean_ci_normal[0], prop_ci_normal[0]],
    "Normal Upper": [mean_ci_normal[1], prop_ci_normal[1]]
})

comparison_table

,Statistic,Bootstrap Lower,Bootstrap Upper,Normal Lower,Normal Upper
0,Mean,149.020613,155.867277,148.979853,155.953845
1,Proportion,0.063333,0.126667,0.060415,0.126252


## Comparison of Bootstrap and Classical Confidence Intervals

The bootstrap and classical intervals are generally similar for the mean and the proportion because both statistics have well-known sampling behavior and standard error formulas. In this case, the interval limits are close, which shows that the normal approximation works well when the sample size is large enough.

Small differences can still appear because the bootstrap interval is built directly from repeated resampling, while the classical interval assumes an ideal theoretical distribution. The bootstrap can therefore reflect slight asymmetry or irregularity in the data that the normal approximation does not capture.

The bootstrap approach is especially useful for the median because there is no simple standard error formula for the median like there is for the mean or the proportion. By resampling many times and recalculating the median, bootstrap gives an empirical estimate of uncertainty without relying on strong distribution assumptions.